# Getting Started Guide

Welcome! This guide will walk you through setting up and connecting to your devices quickly and easily.

If you haven’t installed **InsPy** yet, please follow these instructions:

1. Go to the InsPy installation page: [Installation Guide](../index.md#installation).
2. Follow the on-screen instructions to install InsPy on your device.

## Import the module:
Firstly, create a python file and import the instrument class you need. You can import like the following example:

In [1]:
from inspy.instrument.powersupply import PsuKeysightE36312A

## Get the address of the instrument:

If you already know how to do get the address, head over to [Instantiate the class](#Instantiate-the-class:) section below.

To get the address of the instrument, we need to install **Keysight IO Libraries Suite**. After installing, we can obtain the address from the application depending on the connection type.

The steps for obtaining the address are explained in detail below:


### Step 1: Install Keysight IO Libraries Suite

1. **Download the Software**: Go to the [Keysight IO Libraries Suite download page](https://www.keysight.com/find/iosuite).
2. **Run the Installer**: Once downloaded, open the installer file and follow the on-screen instructions to complete the installation.
3. **Open Keysight IO Libraries Suite**: After installation, launch the software. It will open the main interface with access to **Keysight Connection Expert**.


### Step 2: Obtain Instrument Address Using Keysight Connection Expert

1. **Open Keysight Connection Expert**: Inside Keysight IO Libraries Suite, click on **Keysight Connection Expert**.
2. **Auto-Discover Instruments**: Connection Expert will automatically scan for all connected instruments and display them in the "Instrument I/O on this PC" list.
3. **Locate Your Instrument**: Each detected instrument will show its address based on the connection type (USB, LAN, GPIB, etc.).

**Instrument not showing up**: If your instrument is not showing up and you are using a USB connection, go to `Step 2.1`.
**Connecting a LAN device**: If you want to connect a device via LAN, go to `Step 2.2`.


#### Step 2.1: Manually add USB device:

**Note**: If you are running InsPy on Linux then you need an extra step to get a USB device to work.

``` bash
sudo tee /etc/udev/rules.d/99-inspy.rules > /dev/null <<'EOF'
SUBSYSTEM=="usb", ATTRS{idVendor}=="xxxx", ATTRS{idProduct}=="xxxx", MODE="0660", GROUP="plugdev"
EOF
sudo usermod -aG plugdev "$USER"
sudo udevadm control --reload-rules && sudo udevadm trigger
```

1. **Locate the COM Port your device is connected to**:
   - It should show you a Windows notification with the port number when you connect the device. If you missed it then go to next point.
   - Open **Device Manager** on your computer and locate the **Ports (COM & LPT)** section.
   - Find the COM port that corresponds to your instrument (It is usually shown as USB Serial Device).

2. **Add the instrument on Keysight Connection Expert**: 
   - Open **Keysight Connection Expert** and under **My Instruments** there should be some COM (ASRLx) options. Select the ASRLx that corresponds to the COMx identified before. For example, if you saw your device is connected to COM4, click on COM (ASRL4).
   - Without changing anything, click on `+ Instrument` and then `OK`.
   - Now you should be able to follow the next steps.


#### Step 2.2: Add LAN device:
1. **Make sure the device is connected to the network**
   - If you connect your device via LAN directly to your PC, it is a complicated process and is out of scope for this guide.
   - The easiest thing is to connect it to the network and access via IP address or use a USB connection.

2. **Add the instrument on Keysight Connection Expert**:
   - Open **Keysight Connection Expert** and under **My Instruments** there should be an option for **LAN (TCPIP0)**. Click on that.
   - Click on `+ Instrument`. For most instruments, the device should show under **Select from List**. Just select that device and click on `OK`.
   - If it is not present, click on **Enter Address**.
   - Find the IP address of the device by either going to `Utility -> I/O` (Present for most devices) or look at the manual to figure out how to get the IP address.
   - Once you have the IP address, enter the address in **Hostname or IP Address** field and click on `OK`.


### Step 3: Identify the Instrument Address

1. USB Instruments
   - Just click on the device and you should be able to see the address under **VISA Address**
   - Click on the address and copy it.
   - The address can be of the form `ASRL16::INSTR` or `USB0::0x2A8D::0x1797::CN57046469::0::INSTR`

2. LAN (Ethernet) Instruments
   - Just click on the device and you should be able to see the address under **VISA Address**
   - Click on the address and copy it.
   - The address should be of the form `TCPIP0::172.18.44.89::5025::SOCKET` or some other form.
   - You can also get the address directly from the IP Address. Search the IP address on your browser and you should be able to see the addresses you need.

3. Serial (RS232) Instruments
   - **RS232 Instruments will be added soon**

4. GPIB Instruments
   - **Currently GPIB instruments are out of scope**

If you are still facing issues, please let one of the team members know.


After getting the address, store it in a variable for usage later. We are using a USB connector for the example:

In [2]:
address = "USB0::10893::4354::MY61004144::0::INSTR"

## Instantiate the class:

After you identify the address, you need to make an instance of the classes like the following (Use whatever name you want):

In [3]:
psu = PsuKeysightE36312A(address)

The instrument builds the right transport from the address for you. If you need to customise the connection (e.g. a specific serial setting), you can still pass a transport instance explicitly with the `transport` keyword:

In [ ]:
from inspy.instrument.signalgenerator import SigGenHP33120A
from inspy.transport.serial_transport import SerialTransport

siggen = SigGenHP33120A(
    transport=SerialTransport(
        visa_resource_name="enter_siggen_address_here", baudrate=9600
    )
)

## Connecting to the instrument via InsPy:

The above line will make an object of the instrument but will not connect to it yet.
There are a few ways you can connect to the instrument. Examples are given below. You can use any of them as you like. If they throw an error, your device isn't connected.

1. **Using a context manager (Recommended)**:
The benefit of this is that you don't need to manually connect or disconnect. It will do it automatically.

In [5]:
with psu as _psu:
    print(_psu.get_id())

Keysight Technologies,E36312A,MY61004144,2.1.0-1.0.4-1.12


2. **Using connect and disconnect**: For this you have to connect, then use the instrument and then again disconnect manually. If something goes wrong, you have to make sure it is still calling the disconnect function. For this complexity, it is recommended to use the first approach. Remember to `Disconnect`!

In [6]:
psu.connect()
print(psu.get_id())
psu.disconnect()

Keysight Technologies,E36312A,MY61004144,2.1.0-1.0.4-1.12


Now we can use more useful functions. But before that let's learn how to get help or more information about what you are dealing with.

## Getting Help:

To get information about the class, type either of the following commands, and they should provide information about what you're dealing with. For example:

In [7]:
help(psu)

# or

help(PsuKeysightE36312A)

Help on PsuKeysightE36312A in module inspy.instrument.powersupply.keysight_psu object:

class PsuKeysightE36312A(PsuKeysightBase)
 |  PsuKeysightE36312A(visa_address: str | None = None, transport: inspy.transport.pyvisa_transport.PyVisaTransport | None = None) -> None
 |  
 |  Power Supply instrument class for the Keysight E36312A.
 |  
 |  This power supply has 3 channels
 |  
 |  Channels 1,2,3 refer to P6V, P25V, and N25V respectively
 |  # Voltage trip limits: 6.6V,27.5V,27.5V
 |  # Current Limits: 5A,1A,1A
 |  
 |  `Keysight E36300 User Manual <https://www.keysight.com/gb/en/assets/9018-04577/programming-guides/9018-04577.pdf>`_
 |  
 |  Method resolution order:
 |      PsuKeysightE36312A
 |      PsuKeysightBase
 |      inspy.instrument._base.VisaInstrumentMixin
 |      inspy.instrument.powersupply._base.PowerSupply
 |      inspy.instrument._base.InstrumentBase
 |      builtins.object
 |  
 |  Data and other attributes defined here:
 |  
 |  __abstractmethods__ = frozenset()
 |  


You can also extract the methods that are supported by the instrument:

In [8]:
from inspy import list_methods

print(list_methods(psu))

['await_completion', 'connect', 'disconnect', 'get_channel_state', 'get_current', 'get_current_protection_state', 'get_current_trip_status', 'get_id', 'get_output_voltage_trip_point', 'get_voltage', 'get_voltage_trip_status', 'measure_current', 'measure_voltage', 'reset', 'reset_current_trip_with_disable', 'reset_voltage_trip_with_disable', 'set_channel_state', 'set_current', 'set_current_protection_state', 'set_output_voltage_trip_point', 'set_voltage']


You can also get more information about the method you are going to use. For example:

In [9]:
help(psu.set_output_voltage_trip_point)

Help on method set_output_voltage_trip_point in module inspy.instrument.powersupply.keysight_psu:

set_output_voltage_trip_point(voltage: float, channel: int = 1) -> None method of inspy.instrument.powersupply.keysight_psu.PsuKeysightE36312A instance
    Set the output voltage value at which to trip the power supply.
    
    Args:
        voltage: Voltage trip point.
        channel: Channel to set.



## Examples:

Now that we have gone through the basics, some examples of small power supply programs can be found [here](power_supply.ipynb).

### Notes:

If something is giving a timeout or any error that doesn't look right, try using:

> _psu.await_completion()

If that still doesn't work, you could try:

> time.sleep(0.5)

In rare cases, all your tests might fail. If that happens, it's either your IDE's problem or the device. We suggest restarting the IDE and reconnecting the device.

## All Done!

Now you can create your own program and utilise it as you need